# Step 1: Data Acquisition Pipeline
## Paper: Emergent risk asymmetry in high-dimensional multi-agent self-play
**Submission:** Nature Machine Intelligence

---

**Author:** Kenny Ching  
**Affiliation:** School of Business, University of Auckland  
**Date:** February 2026  

---

**Description:**
This notebook handles the retrieval of raw match replay data from the OpenDota API. It targets two specific cohorts:
1.  **Treatment Group:** The OpenAI Five benchmark matches (2018/2019).
2.  **Control Group:** High-ELO human matches from The International (TI6–TI10).

**Output:**
* Saves raw `.json` replay files to the `data/raw/` directory.

**Notes for Reviewers:**
* **Runtime:** This process involves downloading ~2,300 matches. Due to API rate limiting, this script may take several hours to complete.
* **API Key:** The script uses the public tier of the OpenDota API. No key is required, but rate limits apply.

In [ ]:


import os
import json
import time
import requests
from tqdm import tqdm
from requests.adapters import HTTPAdapter, Retry

# --- CONFIGURATION ---
# Where to save the raw JSONs
DATA_DIR = "data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

# 1. OpenAI Five Match IDs (The Treatment Group)
OPENAI_IDS = [4693543125, 4693543123, 4693543117, 4080601137, 4080420268, 4080316101]

# 2. League IDs for Human Control Group (The International 2016-2021)
TI_LEAGUES = {
    "TI6": 4664,
    "TI7": 5401,
    "TI8": 9870,
    "TI9": 10749,
    "TI10": 13256
}

# --- SETUP SESSION ---
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

def fetch_match_ids(league_id):
    """Get all match IDs for a specific league."""
    print(f"Fetching match list for League {league_id}...")
    try:
        resp = session.get(f"https://api.opendota.com/api/leagues/{league_id}/matches")
        if resp.status_code == 200:
            return [m['match_id'] for m in resp.json()]
    except Exception as e:
        print(f"Error fetching league {league_id}: {e}")
    return []

def download_match(match_id):
    """Download full match details and cache to disk."""
    path = os.path.join(DATA_DIR, f"{match_id}.json")
    if os.path.exists(path):
        return # Skip if exists

    try:
        # We need the 'teamfights' and 'radiant_gold_adv' fields
        resp = session.get(f"https://api.opendota.com/api/matches/{match_id}", timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            # Only save if it has the parsing data we need
            if 'radiant_gold_adv' in data and data['radiant_gold_adv']:
                with open(path, 'w') as f:
                    json.dump(data, f)
            else:
                pass # Skip unparsed matches
        time.sleep(0.8) # Respect rate limits
    except Exception as e:
        print(f"Failed to download {match_id}: {e}")

if __name__ == "__main__":
    # A. Gather IDs
    all_match_ids = set(OPENAI_IDS)
    print("Gathering Human Baseline Match IDs...")
    for league_name, lid in TI_LEAGUES.items():
        ids = fetch_match_ids(lid)
        print(f"  > {league_name}: Found {len(ids)} matches")
        all_match_ids.update(ids)

    print(f"Total Matches to Download: {len(all_match_ids)}")

    # B. Download Loop
    print(f"Downloading to {DATA_DIR}...")
    for mid in tqdm(list(all_match_ids)):
        download_match(mid)

    print("Download Complete.")